<a href="https://colab.research.google.com/github/Diwash17/Mistral-7B-Fine-tuning-with-LoRA-QLoRA/blob/main/mistralfinetunning_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mistral-7B Fine-tuning with LoRA & QLoRA
Healthcare chatbot fine-tuning on HealthCareMagic dataset

In [1]:
!nvidia-smi

Sat Jul 11 04:20:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl gradio huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.0 MB/s eta 0:00:00


In [3]:
import transformers
import datasets
import peft
import bitsandbytes

In [4]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [6]:
from datasets import load_dataset

# Load HealthCareMagic healthcare dataset
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")

print(dataset)
print("\nFirst example:")
print(dataset['train'][0])

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 112165
    })
})

First example:
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of perip

In [7]:
def format_instruction(example):
    """
    Format dataset examples into instruction style
    for Mistral fine-tuning.
    """
    return {
        "text": f"""<s>[INST] You are a helpful medical assistant.
{example['instruction']}

Patient: {example['input']} [/INST]

Doctor: {example['output']} </s>"""
    }

# Apply formatting to entire dataset
formatted_dataset = dataset.map(format_instruction)

print("Formatting done! ✅")
print("\nFirst formatted example:")
print(formatted_dataset['train'][0]['text'])

Map:   0%|          | 0/112165 [00:00<?, ? examples/s]

Formatting done! ✅

First formatted example:
<s>[INST] You are a helpful medical assistant.
If you are a doctor, please answer the medical questions based on the patient's description.

Patient: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc! [/INST]

Doctor: Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is di

In [8]:
def is_valid_example(example):
    """
    Filter out low quality examples from dataset.
    """
    # Remove empty inputs or outputs
    if not example['input'] or not example['output']:
        return False

    # Remove very short responses (low quality)
    if len(example['output'].split()) < 20:
        return False

    # Remove very long responses (memory issues on Colab)
    if len(example['text'].split()) > 512:
        return False

    return True

# Apply filtering
cleaned_dataset = formatted_dataset.filter(is_valid_example)

print(f"Original dataset size: {len(formatted_dataset['train'])}")
print(f"Cleaned dataset size: {len(cleaned_dataset['train'])}")
print(f"Removed: {len(formatted_dataset['train']) - len(cleaned_dataset['train'])} examples")

Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

Original dataset size: 112165
Cleaned dataset size: 109792
Removed: 2373 examples


In [9]:
# Split dataset into train and test
split_dataset = cleaned_dataset['train'].train_test_split(
    test_size=0.2,  # 20% for testing
    seed=42         # for reproducibility
)

print(split_dataset)
print(f"\nTraining examples: {len(split_dataset['train'])}")
print(f"Testing examples: {len(split_dataset['test'])}")

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 87833
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 21959
    })
})

Training examples: 87833
Testing examples: 21959


In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model name
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")
print(f"Model device: {next(model.parameters()).device}")

Loading tokenizer...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully!
Model device: cuda:0


## Tokenize Dataset

In [11]:
# Tokenize the dataset
def tokenize_function(examples):
    """
    Tokenize the text examples.
    """
    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        max_length=512,
        truncation=True,
        return_tensors=None
    )

    # Set labels to input_ids for language modeling
    tokenized['labels'] = tokenized['input_ids'].copy()

    return tokenized

print("Tokenizing training dataset...")
tokenized_train = split_dataset['train'].map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=['text']
)

print("Tokenizing test dataset...")
tokenized_test = split_dataset['test'].map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=['text']
)

print("✅ Tokenization complete!")
print(f"Train dataset: {len(tokenized_train)}")
print(f"Test dataset: {len(tokenized_test)}")
print(f"\nExample tokens: {tokenized_train[0]}")

Tokenizing training dataset...


Map:   0%|          | 0/87833 [00:00<?, ? examples/s]

Tokenizing test dataset...


Map:   0%|          | 0/21959 [00:00<?, ? examples/s]

✅ Tokenization complete!
Train dataset: 87833
Test dataset: 21959

Example tokens: {'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'what happens when someone has low potassium?', 'output': 'Hi. Hypokalemia (low potassium) can produce symptoms on heart and muscles... It can cause arrhythmias and other conduction abnormalities on the heart. It can produce muscle weakness resulting in flaccidity and paralysis of the muscles, difficulty in passing urine, constipation. On severe potassium loss can result in breathing difficulties. Cardiac arrhythmias can be life threatening condition and needs immediate attention.', 'input_ids': [1, 1, 28792, 16289, 28793, 995, 460, 264, 10865, 5714, 13892, 28723, 13, 3381, 368, 460, 264, 6676, 28725, 4665, 4372, 272, 5714, 4224, 2818, 356, 272, 7749, 28742, 28713, 5436, 28723, 13, 13, 8243, 722, 28747, 767, 6881, 739, 2493, 659, 2859, 2513, 489, 1962, 28804, 733, 28748, 16289, 28793, 

## QLoRA Fine-tuning Configuration

In [12]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA config for QLoRA fine-tuning
lora_config = LoraConfig(
    r=8,                           # LoRA rank
    lora_alpha=16,                 # LoRA alpha (2x rank is common)
    target_modules=["q_proj", "v_proj"],  # Target attention modules
    lora_dropout=0.05,             # Dropout for LoRA layers
    bias="none",                   # No bias adaptation
    task_type="CAUSAL_LM"
)

print("LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Target modules: {lora_config.target_modules}")

# Get PEFT model
model = get_peft_model(model, lora_config)

# Print trainable parameters
def print_trainable_parameters(model):
    """
    Print number of trainable parameters.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params:,d} || all params: {all_param:,d} || trainable%: {100 * trainable_params / all_param:.2f}")

print("\n📊 Model Parameters:")
print_trainable_parameters(model)

LoRA Configuration:
  Rank (r): 8
  Alpha: 16
  Dropout: 0.05
  Target modules: {'q_proj', 'v_proj'}

📊 Model Parameters:
trainable params: 3,407,872 || all params: 3,755,479,040 || trainable%: 0.09


## Training Configuration

In [17]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import os

output_dir = "./mistral-7b-healthcare-qlora"
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    fp16=True,
    seed=42,
)

## Initialize Trainer

In [18]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal language modeling (not masked)
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    callbacks=[
        # Early stopping callback (optional)
        # EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)
    ]
)

print("✅ Trainer initialized successfully!")

✅ Trainer initialized successfully!


## Start Fine-tuning

In [19]:
# Start training
print("🔥 Starting QLoRA fine-tuning...")
train_result = trainer.train()

print("\n✅ Training completed!")
print(f"Training loss: {train_result.training_loss:.4f}")

🔥 Starting QLoRA fine-tuning...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## Evaluate Model

In [ ]:
# Evaluate on test set
print("📊 Evaluating on test set...")
eval_result = trainer.evaluate()

print("\nEvaluation Results:")
print(f"  Eval Loss: {eval_result['eval_loss']:.4f}")
print(f"  Perplexity: {eval_result.get('eval_perplexity', 'N/A')}")

## Save Model

In [ ]:
# Save the LoRA adapter
print("💾 Saving LoRA adapter...")
model.save_pretrained(f"{output_dir}/final-adapter")

# Save tokenizer
tokenizer.save_pretrained(f"{output_dir}/final-adapter")

print(f"✅ Model saved to {output_dir}/final-adapter")

## Load and Merge Model (Optional)

In [ ]:
from peft import AutoPeftModelForCausalLM

# Load the trained LoRA model
model_path = f"{output_dir}/final-adapter"
qlora_model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16
)

# Merge LoRA weights with base model (if you want a single model file)
print("Merging LoRA weights with base model...")
merged_model = qlora_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(f"{output_dir}/merged-model")
tokenizer.save_pretrained(f"{output_dir}/merged-model")

print(f"✅ Merged model saved to {output_dir}/merged-model")

## Test Inference

In [ ]:
# Test inference with the fine-tuned model
from transformers import pipeline

# Load the model for inference
generator = pipeline(
    'text-generation',
    model=merged_model,
    tokenizer=tokenizer,
    device=0  # Use GPU if available
)

# Test prompt
test_prompt = """<s>[INST] You are a helpful medical assistant.
I have been experiencing persistent headaches for the past week. What could be the cause?

Patient: I wake up with a throbbing headache every morning. [/INST]

Doctor:"""

print("Testing inference...\n")
response = generator(
    test_prompt,
    max_length=300,
    temperature=0.7,
    top_p=0.9,
    num_return_sequences=1
)

print("Generated Response:")
print(response[0]['generated_text'][len(test_prompt):])

## Upload to Hugging Face Hub (Optional)

In [ ]:
# Upload the adapter to Hugging Face Hub
repo_name = "mistral-7b-healthcare-qlora"

print(f"Uploading to Hub as {repo_name}...")
model.push_to_hub(repo_name, use_auth_token=True)

print(f"✅ Model uploaded to https://huggingface.co/your_username/{repo_name}")

## Alternative: LoRA (without quantization) Configuration

For reference, here's how to use LoRA without quantization (requires more VRAM):

```python
# Load model in full precision (requires more GPU memory)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,  # or torch.bfloat16
    trust_remote_code=True
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Same LoRA config can be used
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
# Rest of training remains the same
```